In [ ]:
import random
import time
from collections import Counter

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "sentence-transformers/paraphrase-MiniLM-L3-v2"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 256 if device == "mps" else 64
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["sentence1"] = df["sentence1"].astype(str)
df["sentence2"] = df["sentence2"].astype(str)
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

all_sentences = sentences1 + sentences2
unique_sentences = list(dict.fromkeys(all_sentences))
sentence_to_idx = {text: idx for idx, text in enumerate(unique_sentences)}

pair_idx1 = np.fromiter((sentence_to_idx[s] for s in sentences1), dtype=np.int32, count=len(sentences1))
pair_idx2 = np.fromiter((sentence_to_idx[s] for s in sentences2), dtype=np.int32, count=len(sentences2))

total_sentence_occurrences = len(all_sentences)
num_unique_sentences = len(unique_sentences)
cache_hits = total_sentence_occurrences - num_unique_sentences
cache_hit_rate = cache_hits / total_sentence_occurrences if total_sentence_occurrences else 0.0

sentence1_counter = Counter(sentences1)
sentence2_counter = Counter(sentences2)
role_overlap_count = sum(1 for s in unique_sentences if s in sentence1_counter and s in sentence2_counter)

print({
    "total_sentence_occurrences": total_sentence_occurrences,
    "num_unique_sentences": num_unique_sentences,
    "cache_hits": cache_hits,
    "cache_hit_rate": round(cache_hit_rate, 6),
    "num_unique_sentence1": len(sentence1_counter),
    "num_unique_sentence2": len(sentence2_counter),
    "num_sentences_seen_in_both_roles": role_overlap_count,
})

In [ ]:
model = SentenceTransformer(model_name, device=device)
model.eval()

unique_embeddings = model.encode(
    unique_sentences,
    batch_size=batch_size,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)

print({
    "model_name": model_name,
    "embedding_shape": tuple(unique_embeddings.shape),
})

In [ ]:
emb1 = unique_embeddings[pair_idx1]
emb2 = unique_embeddings[pair_idx2]

cosine_similarity = np.sum(emb1 * emb2, axis=1)
predicted_score_0_5 = 2.5 * (cosine_similarity + 1.0)
error = predicted_score_0_5 - labels
abs_error = np.abs(error)

results_df = df.copy()
results_df["sentence1_idx"] = pair_idx1
results_df["sentence2_idx"] = pair_idx2
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["error"] = error
results_df["abs_error"] = abs_error

print(results_df[["sentence1", "sentence2", "label", "cosine_similarity", "predicted_score_0_5", "error"]].head(10))

In [ ]:
sentence1_role_stats = (
    results_df.groupby("sentence1")
    .agg(
        occurrences=("sentence1", "size"),
        mean_label=("label", "mean"),
        min_label=("label", "min"),
        max_label=("label", "max"),
        mean_predicted_score=("predicted_score_0_5", "mean"),
        min_predicted_score=("predicted_score_0_5", "min"),
        max_predicted_score=("predicted_score_0_5", "max"),
    )
    .reset_index()
    .rename(columns={"sentence1": "sentence"})
)
sentence1_role_stats = sentence1_role_stats[sentence1_role_stats["occurrences"] > 1].copy()
sentence1_role_stats["predicted_score_range"] = sentence1_role_stats["max_predicted_score"] - sentence1_role_stats["min_predicted_score"]
sentence1_role_stats["label_range"] = sentence1_role_stats["max_label"] - sentence1_role_stats["min_label"]
sentence1_role_stats = sentence1_role_stats.sort_values(
    by=["occurrences", "predicted_score_range", "sentence"], ascending=[False, False, True]
).reset_index(drop=True)

sentence2_role_stats = (
    results_df.groupby("sentence2")
    .agg(
        occurrences=("sentence2", "size"),
        mean_label=("label", "mean"),
        min_label=("label", "min"),
        max_label=("label", "max"),
        mean_predicted_score=("predicted_score_0_5", "mean"),
        min_predicted_score=("predicted_score_0_5", "min"),
        max_predicted_score=("predicted_score_0_5", "max"),
    )
    .reset_index()
    .rename(columns={"sentence2": "sentence"})
)
sentence2_role_stats = sentence2_role_stats[sentence2_role_stats["occurrences"] > 1].copy()
sentence2_role_stats["predicted_score_range"] = sentence2_role_stats["max_predicted_score"] - sentence2_role_stats["min_predicted_score"]
sentence2_role_stats["label_range"] = sentence2_role_stats["max_label"] - sentence2_role_stats["min_label"]
sentence2_role_stats = sentence2_role_stats.sort_values(
    by=["occurrences", "predicted_score_range", "sentence"], ascending=[False, False, True]
).reset_index(drop=True)

print("sentence1_role_stats_top10")
print(sentence1_role_stats.head(10))
print("sentence2_role_stats_top10")
print(sentence2_role_stats.head(10))

In [ ]:
pearson_corr = pearsonr(predicted_score_0_5, labels).statistic
spearman_corr = spearmanr(predicted_score_0_5, labels).statistic

results_df["label_decile"] = pd.qcut(results_df["label"], q=10, labels=False, duplicates="drop")
calibration_df = (
    results_df.groupby("label_decile", dropna=False)
    .agg(
        count=("label", "size"),
        label_min=("label", "min"),
        label_max=("label", "max"),
        label_mean=("label", "mean"),
        prediction_mean=("predicted_score_0_5", "mean"),
        prediction_std=("predicted_score_0_5", "std"),
        mean_error=("error", "mean"),
        mean_abs_error=("abs_error", "mean"),
    )
    .reset_index()
)

largest_overpredictions = results_df.sort_values(
    by=["error", "abs_error"], ascending=[False, False]
).head(10).reset_index(drop=True)

largest_underpredictions = results_df.sort_values(
    by=["error", "abs_error"], ascending=[True, False]
).head(10).reset_index(drop=True)

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
})
print("calibration_by_label_decile")
print(calibration_df)
print("largest_overpredictions")
print(largest_overpredictions[["sentence1", "sentence2", "label", "predicted_score_0_5", "error"]])
print("largest_underpredictions")
print(largest_underpredictions[["sentence1", "sentence2", "label", "predicted_score_0_5", "error"]])

In [ ]:
runtime_seconds = time.time() - start_time

print(f"device_used: {device}")
print(f"model_name: {model_name}")
print(f"dataset_split: {dataset_name}/{dataset_config}/{split_name}")
print(f"num_examples: {len(df)}")
print(f"total_sentence_occurrences: {total_sentence_occurrences}")
print(f"num_unique_sentences: {num_unique_sentences}")
print(f"cache_hits: {cache_hits}")
print(f"cache_hit_rate: {cache_hit_rate:.6f}")
print(f"num_unique_sentence1: {len(sentence1_counter)}")
print(f"num_unique_sentence2: {len(sentence2_counter)}")
print(f"num_sentences_seen_in_both_roles: {role_overlap_count}")
print(f"pearson_correlation: {pearson_corr:.6f}")
print(f"spearman_correlation: {spearman_corr:.6f}")
print(f"num_repeated_sentence1_entries: {len(sentence1_role_stats)}")
print(f"num_repeated_sentence2_entries: {len(sentence2_role_stats)}")
print("sentence1_role_top10:")
print(sentence1_role_stats.head(10).to_dict(orient="records"))
print("sentence2_role_top10:")
print(sentence2_role_stats.head(10).to_dict(orient="records"))
print("calibration_by_label_decile:")
print(calibration_df.to_dict(orient="records"))
print("largest_overpredictions:")
print(largest_overpredictions[["sentence1", "sentence2", "label", "predicted_score_0_5", "error"]].to_dict(orient="records"))
print("largest_underpredictions:")
print(largest_underpredictions[["sentence1", "sentence2", "label", "predicted_score_0_5", "error"]].to_dict(orient="records"))
print(f"runtime_seconds: {runtime_seconds:.2f}")